In [ ]:
# nnU-Net 肺栓塞分割完整流程
# 以调用现有脚本的方式整合所有步骤

import subprocess
import sys
import os

# ===================== 配置 =====================
SCRIPTS_DIR = "/root/autodl-tmp/Project/nnUNet/scripts"

# ===================== 辅助函数 =====================
def run_python_script(script_name, description):
    """运行 Python 脚本"""
    print(f"\n{'='*50}")
    print(f"▶ {description}")
    print(f"{'='*50}")
    script_path = os.path.join(SCRIPTS_DIR, script_name)
    result = subprocess.run([sys.executable, script_path], capture_output=False)
    if result.returncode != 0:
        print(f"❌ {description} 失败")
        return False
    print(f"✅ {description} 完成")
    return True

def run_bash_script(script_name, description, menu_choice=None):
    """运行 Bash 脚本，可选自动选择菜单项"""
    print(f"\n{'='*50}")
    print(f"▶ {description}")
    print(f"{'='*50}")
    script_path = os.path.join(SCRIPTS_DIR, script_name)
    if menu_choice is not None:
        # 通过 echo 自动输入菜单选项
        cmd = f'echo "{menu_choice}" | bash "{script_path}"'
        result = subprocess.run(cmd, shell=True, capture_output=False)
    else:
        result = subprocess.run(["bash", script_path], capture_output=False)
    if result.returncode != 0:
        print(f"❌ {description} 失败")
        return False
    print(f"✅ {description} 完成")
    return True

def confirm(message):
    """用户确认"""
    response = input(f"{message} (y/n): ").strip().lower()
    return response == 'y'

# ===================== 主流程 =====================
print("""
╔══════════════════════════════════════════╗
║   nnU-Net 肺栓塞分割 Pipeline           ║
║   以调用现有脚本的方式整合所有步骤      ║
╚══════════════════════════════════════════╝
""")

print("可用步骤：")
print("  1. DICOM 转 NIfTI (step1)")
print("  2. 构建数据集 (step2)")
print("  3. 转换数据集 (convert_MSD_dataset)")
print("  4. 计划与预处理 (plan_and_preprocess)")
print("  5. 训练 (train)")
print("  6. 预测 (predict)")
print("  7. 检查标签")
print("  8. 校验数据集")
print("  9. 一键跑完 全部")
print("  10. 清理数据 (clear.sh)")
print("  0. 退出")

choice = input("\n请选择步骤 [0-10]: ").strip()

if choice == '0':
    sys.exit(0)

elif choice == '1':
    run_python_script('convert_1.py', 'Step 1: DICOM 转 NIfTI')

elif choice == '2':
    run_python_script('convert_2.py', 'Step 2: 构建数据集')

elif choice == '3':
    run_bash_script('convert_3.sh', 'Step 3: 转换数据集', '1')

elif choice == '4':
    run_bash_script('convert_3.sh', 'Step 4: 计划与预处理', '2')

elif choice == '5':
    run_bash_script('convert_3.sh', 'Step 5: 训练', '3')

elif choice == '6':
    run_bash_script('convert_3.sh', 'Step 6: 预测', '4')

elif choice == '7':
    run_bash_script('convert_3.sh', 'Step 7: 检查标签', '7')

elif choice == '8':
    run_bash_script('convert_3.sh', 'Step 8: 校验数据集', '6')

elif choice == '9':
    print("\n🚀 一键跑完完整流程：DICOM → 训练")
    if confirm("确认开始？"):
        steps = [
            (run_python_script, 'convert_1.py', 'Step 1: DICOM 转 NIfTI'),
            (run_python_script, 'convert_2.py', 'Step 2: 构建数据集'),
            (run_bash_script, 'convert_3.sh', 'Step 3: 转换数据集', '1'),
            (run_bash_script, 'convert_3.sh', 'Step 4: 计划与预处理', '2'),
            (run_bash_script, 'convert_3.sh', 'Step 5: 训练', '3'),
        ]
        for step in steps:
            if step[0] == run_python_script:
                success = run_python_script(step[1], step[2])
            else:
                success = run_bash_script(step[1], step[2], step[3])
            if not success:
                print("⚠️ 流程中断")
                break
        else:
            print("\n🎉 全部完成！")
    else:
        print("已取消")

elif choice == '10':
    print("\n▶ 打开清理菜单")
    subprocess.run(["bash", os.path.join(SCRIPTS_DIR, 'clear.sh')])

else:
    print("无效选项")